# Module 6.4 — Local Deployment: Ollama + Full Local Pipeline

**Goal:** Import DeskMate's GGUF (Module 5.3) into Ollama, write a custom Modelfile with DeskMate's system prompt and generation parameters, and run the full pipeline (encoder + FAISS retriever + Ollama decoder) entirely offline — no GPU, no internet.

## 1. Prerequisites

In [ ]:
# Ollama must be installed: https://ollama.com/download
# !curl -fsSL https://ollama.com/install.sh | sh

# !pip install -q ollama openai rouge-score sentence-transformers faiss-cpu

import os, time, json, subprocess, statistics
import numpy as np
import matplotlib.pyplot as plt

SMOKE_TEST  = True   # set False to run real Ollama commands
GGUF_PATH   = 'models/deskmate-Q4_K_M.gguf'
MODEL_NAME  = 'deskmate'
OLLAMA_URL  = 'http://localhost:11434'

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Write the Modelfile

In [ ]:
MODELFILE_CONTENT = '''
FROM ./models/deskmate-Q4_K_M.gguf

SYSTEM """
You are DeskMate, a concise support assistant.
Answer the ticket using ONLY the provided context passages.
Cite each claim with [Source N] where N is the passage number.
If the context does not contain the answer, respond with:
I do not have that information - please contact support@deskmate.com.
"""

PARAMETER temperature 0.0
PARAMETER num_predict 200
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
'''.strip()

with open('Modelfile', 'w') as f:
    f.write(MODELFILE_CONTENT)

print('Modelfile written:')
print(MODELFILE_CONTENT)

## 3. Build DeskMate in Ollama

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping ollama create.')
    print('[Simulated] ollama create deskmate -f Modelfile')
    print('[Simulated] deskmate model registered in local Ollama registry.')
else:
    result = subprocess.run(
        ['ollama', 'create', MODEL_NAME, '-f', 'Modelfile'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

    # List models
    ls = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    print('\nInstalled models:')
    print(ls.stdout)

## 4. Test via Ollama Python SDK

In [ ]:
GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'ref': 'Contact support with your invoice numbers. We will investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my 2FA device?',
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'ref': 'This is a known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
    {'ticket': 'Can I get a refund after 30 days?',
     'ref': 'DeskMate offers a 30-day money-back guarantee. Refunds after 30 days are case by case.'},
    {'ticket': 'My iOS app crashes on iPhone 14.',
     'ref': 'A crash affecting iOS 17 was fixed in app version 2.1.3. Please update from the App Store.'},
]

def ollama_chat(ticket, model=MODEL_NAME):
    if SMOKE_TEST:
        return f'[Simulated Ollama reply for: {ticket[:40]}...]'
    import ollama
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': f'Ticket: {ticket}'}],
    )
    return resp['message']['content'].strip()

# Test first example
reply = ollama_chat(GOLD[0]['ticket'])
print('Ticket:', GOLD[0]['ticket'])
print('Reply:', reply)

## 5. Throughput Benchmark (CPU)

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulated CPU throughput.')
    # Simulated: ~12 tok/sec on Intel i7, ~35 tok/sec on M2
    ollama_tps = 12.5
    ollama_p50_ms = round(200 / ollama_tps * 1000)
else:
    import ollama
    times = []
    for ex in GOLD * 2:
        t0 = time.perf_counter()
        ollama.chat(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': f"Ticket: {ex['ticket']}"}],
        )
        times.append(time.perf_counter() - t0)
    ollama_p50_ms = round(statistics.median(times) * 1000)
    ollama_tps    = round(200 / statistics.median(times), 1)

print(f'Ollama CPU p50 latency: {ollama_p50_ms} ms')
print(f'Ollama CPU tokens/sec:  {ollama_tps}')

## 6. ROUGE-L vs vLLM Baseline

In [ ]:
from rouge_score import rouge_scorer as rs_module

_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulated ROUGE-L.')
    ollama_rouge = 0.449
    vllm_rouge   = 0.470
    delta = round(ollama_rouge - vllm_rouge, 4)
else:
    scores = []
    for ex in GOLD:
        reply = ollama_chat(ex['ticket'])
        scores.append(_scorer.score(ex['ref'], reply)['rougeL'].fmeasure)
    ollama_rouge = round(sum(scores) / len(scores), 4)
    vllm_rouge   = 0.470  # from Module 6.2
    delta = round(ollama_rouge - vllm_rouge, 4)

print(f'Ollama (Q4_K_M) ROUGE-L: {ollama_rouge}')
print(f'vLLM fp16 ROUGE-L:        {vllm_rouge}')
print(f'Delta: {delta}')
gate = abs(delta) <= 0.03
print(f'Quality gate (|delta| <= 0.03): {"PASS" if gate else "REVIEW"}')

## 7. OpenAI-Compatible API (Ollama v0.1.24+)

In [ ]:
# Ollama exposes /v1/chat/completions — same interface as vLLM and OpenAI
# This means the same FastAPI service from Module 6.3 works with Ollama
# by just changing VLLM_URL to http://localhost:11434/v1/chat/completions

if SMOKE_TEST:
    print('SMOKE_TEST=True — showing API call pattern.')
else:
    from openai import OpenAI
    client = OpenAI(base_url=f'{OLLAMA_URL}/v1', api_key='ollama')
    r = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': 'Ticket: I was charged twice.'}],
        max_tokens=150, temperature=0.0,
    )
    print(r.choices[0].message.content)

print('Pattern: same OpenAI client, just change base_url to Ollama endpoint.')
print(f'Ollama endpoint: {OLLAMA_URL}/v1')

## 8. Full Local Pipeline (Encoder + FAISS + Ollama)

In [ ]:
# This cell shows the structure of a fully offline DeskMate pipeline.
# Replace the placeholders with your Module 2.5 encoder, Module 4.2 index,
# and Module 4.3 chunk_records.

PIPELINE_CODE = '''
# Full offline DeskMate pipeline
import ollama, faiss, numpy as np
from sentence_transformers import SentenceTransformer

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

# Load local assets (no internet required after initial download)
embedder     = SentenceTransformer("BAAI/bge-small-en-v1.5")
faiss_index  = faiss.read_index("data/deskmate_faq.index")
# chunk_records = loaded from data/chunk_records.jsonl

def answer_ticket_local(ticket):
    # 1. Embed query
    q_emb = embedder.encode(BGE_PREFIX + ticket, normalize_embeddings=True)
    # 2. Retrieve top-3 chunks
    _, ids = faiss_index.search(np.array([q_emb], dtype="float32"), k=3)
    chunks = [chunk_records[i] for i in ids[0] if i >= 0]
    # 3. Build RAG prompt
    context = "\n\n".join(
        f"[Source {i+1}] {c[\"source\"]}\n{c[\"text\"]}" for i, c in enumerate(chunks)
    )
    # 4. Generate via Ollama (local GGUF, CPU)
    resp = ollama.chat(
        model="deskmate",
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nTicket: {ticket}"
        }]
    )
    return resp["message"]["content"].strip()
'''

print('Full local pipeline structure:')
print(PIPELINE_CODE)

## 9. Serving Variants Summary Chart

In [ ]:
vllm_tps   = 470.0   # from Module 6.2
vllm_p50   = 1400    # ms

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

variants   = ['vLLM fp16\n(GPU server)', 'GGUF Q4_K_M\n(Ollama CPU)']
tps_vals   = [vllm_tps, ollama_tps]
p50_vals   = [vllm_p50, ollama_p50_ms]
colors     = ['steelblue', 'coral']

ax = axes[0]
ax.bar(variants, tps_vals, color=colors)
ax.set_ylabel('Tokens / second')
ax.set_title('Throughput')
for i, v in enumerate(tps_vals):
    ax.text(i, v + 3, str(v), ha='center', fontsize=9)

ax = axes[1]
ax.bar(variants, p50_vals, color=colors)
ax.set_ylabel('p50 latency (ms) — lower is better')
ax.set_title('Latency (100-token reply)')
for i, v in enumerate(p50_vals):
    ax.text(i, v + 30, f'{v}ms', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('local_deploy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: local_deploy_comparison.png')

## 10. Save Report

In [ ]:
report = [
    '# Local Deployment Report (Ollama)\n',
    f'Model: {MODEL_NAME} ({GGUF_PATH})',
    f'Smoke test: {SMOKE_TEST}\n',
    '| Metric | vLLM fp16 (GPU) | Ollama Q4_K_M (CPU) |',
    '|--------|----------------|---------------------|',
    f'| Throughput (tok/s) | {vllm_tps} | {ollama_tps} |',
    f'| p50 latency (ms) | {vllm_p50} | {ollama_p50_ms} |',
    f'| ROUGE-L | {vllm_rouge} | {ollama_rouge} |',
    f'| ROUGE-L delta | — | {delta} |',
    f'| Requires GPU | Yes | No |',
    f'| Requires internet | Yes (first load) | No |',
    '',
    '## Checkpoint',
    '',
    'Local deployment is a FEATURE for:',
    '  1. Data privacy: PII in tickets never leaves the device.',
    '  2. Compliance: GDPR/HIPAA may prohibit sending data to cloud APIs.',
    '  3. Offline availability: field service teams and air-gapped environments.',
]

with open('reports/local_deploy_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/local_deploy_report.md')
print('\n=== Module 6.4 Complete ===')